<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/M0_BRIDGE_2048_SOURCE_LOCK_v1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M0_BRIDGE_2048_SOURCE_LOCK_v1.1

**v1 대비 핵심 변경:**

| 항목 | v1 (잘못) | v1.1 (수정) |
|:---|:---|:---|
| Learning 기준 수 | 이전 M0 N=7,534 (부분) | 현재 완전 데이터 8,384 |
| 이전 M0 N | Gate 기준 | `LEGACY_M0_PARTIAL_REFERENCE` 참고만 |
| 전체 기준 | 없음 | 43,369개 일치 필수 Gate |
| Audit gate | `source_files_unchanged` 누락 | 포함 |
| SHA-256 전략 | 전체 full SHA | quick fingerprint → 충돌 후보만 full SHA |
| Checkpoint 위치 | 실행 폴더 (재실행 시 초기화) | `_checkpoints/` 고정 위치 (이어서 처리) |
| 오류 처리 | RuntimeError 중단 | 기록 후 계속 → 상태값 보고 |

## 절대 원칙

- 원본 CSV 수정·삭제·이동·이름 변경 **금지**
- 기존 노트북 (`v1`) 덮어쓰기 **금지**
- M0 기준 파일 변경 **금지**
- 모든 출력: `bridge_outputs/YYYYMMDD_HHMMSS_source_lock_v1_1/`
- Checkpoint: `bridge_outputs/_checkpoints/source_lock_v1_1/` (영구)

## 현재 완전 데이터셋 기준

| Split | 기준 합계 |
|:---|---:|
| LEARNING | 8,384 |
| TEST | 15,462 |
| FULL_TEST | 19,523 |
| **전체** | **43,369** |

## 실행 순서

1. **셀 01** — 단위 검증 (드라이브 마운트 없이 실행 가능)
2. **셀 02** — 메인 실행 (전체 파이프라인)
3. **셀 03** — fatal 오류 표시 (오류 발생 시 확인)

## 재실행 시

Colab 연결이 끊겨도 `_checkpoints/fingerprint_checkpoint.csv`를 읽어
완료된 파일의 지문 계산을 건너뜁니다. 셀 02를 다시 실행하면 됩니다.

In [ ]:
# ================================================================
# 셀 01 — 단위 검증 (드라이브 마운트 불필요, 먼저 실행)
# ================================================================

import re

# ── 1. 설정값 검증 ────────────────────────────────────────────
CURRENT_EXPECTED_CHECK = {
    "LEARNING": {
        "Bearing1_1": 3269, "Bearing1_2": 1015,
        "Bearing2_1": 1062, "Bearing2_2": 797,
        "Bearing3_1": 604,  "Bearing3_2": 1637,
    },
    "TEST": {
        "Bearing1_3": 590,  "Bearing1_4": 1327, "Bearing1_5": 2677,
        "Bearing1_6": 2232, "Bearing1_7": 1752, "Bearing2_3": 1202,
        "Bearing2_4": 713,  "Bearing2_5": 2337, "Bearing2_6": 572,
        "Bearing2_7": 200,  "Bearing3_3": 1860,
    },
    "FULL_TEST": {
        "Bearing1_3": 2375, "Bearing1_4": 1665, "Bearing1_5": 2873,
        "Bearing1_6": 2856, "Bearing1_7": 2635, "Bearing2_3": 1955,
        "Bearing2_4": 876,  "Bearing2_5": 2697, "Bearing2_6": 817,
        "Bearing2_7": 268,  "Bearing3_3": 506,
    },
}

EXPECTED_SPLIT_TOTALS_CHECK = {
    split: sum(v.values())
    for split, v in CURRENT_EXPECTED_CHECK.items()
}

assert EXPECTED_SPLIT_TOTALS_CHECK["LEARNING"]  == 8384,  f"LEARNING  합계 오류: {EXPECTED_SPLIT_TOTALS_CHECK['LEARNING']}"
assert EXPECTED_SPLIT_TOTALS_CHECK["TEST"]      == 15462, f"TEST      합계 오류: {EXPECTED_SPLIT_TOTALS_CHECK['TEST']}"
assert EXPECTED_SPLIT_TOTALS_CHECK["FULL_TEST"] == 19523, f"FULL_TEST 합계 오류: {EXPECTED_SPLIT_TOTALS_CHECK['FULL_TEST']}"

EXPECTED_TOTAL_CHECK = sum(EXPECTED_SPLIT_TOTALS_CHECK.values())
assert EXPECTED_TOTAL_CHECK == 43369, f"전체 합계 오류: {EXPECTED_TOTAL_CHECK}"

EXPECTED_SAMPLES_CHECK = int(25600 * 0.1)
assert EXPECTED_SAMPLES_CHECK == 2560

COL_H_CHECK = 4
assert COL_H_CHECK == 4

LEGACY_LEARNING_TOTAL = sum([
    2803, 871, 911, 797, 515, 1637
])
assert LEGACY_LEARNING_TOTAL == 7534

print("[Unit] 설정값 assert 통과 ✅")
print(f"  LEARNING={EXPECTED_SPLIT_TOTALS_CHECK['LEARNING']:,}  "
      f"TEST={EXPECTED_SPLIT_TOTALS_CHECK['TEST']:,}  "
      f"FULL_TEST={EXPECTED_SPLIT_TOTALS_CHECK['FULL_TEST']:,}  "
      f"TOTAL={EXPECTED_TOTAL_CHECK:,}")
print(f"  LEGACY M0 Learning N={LEGACY_LEARNING_TOTAL:,}  "
      f"현재 기준={EXPECTED_SPLIT_TOTALS_CHECK['LEARNING']:,}  "
      f"차이={EXPECTED_SPLIT_TOTALS_CHECK['LEARNING']-LEGACY_LEARNING_TOTAL:,}")

# ── 2. Natural sort 검증 ─────────────────────────────────────
def _natural_key(s):
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", str(s))]

names = ["acc_10.csv", "acc_2.csv", "acc_1.csv", "acc_200.csv"]
sorted_names = sorted(names, key=_natural_key)
assert sorted_names == ["acc_1.csv", "acc_2.csv", "acc_10.csv", "acc_200.csv"], \
    f"natural sort 실패: {sorted_names}"
print("[Unit] natural sort 통과 ✅")

# ── 3. COMPILE 검증 (메인 코드 구문 오류 사전 탐지) ──────────
# 이 셀에서는 메인 코드를 문자열로 compile만 시도하지 않음
# (코드가 이미 별도 셀에 있으므로 Colab이 자동 파싱)
print("[Unit] 모든 단위 검증 통과 ✅")

[Unit] 설정값 assert 통과 ✅
  LEARNING=8,384  TEST=15,462  FULL_TEST=19,523  TOTAL=43,369
  LEGACY M0 Learning N=7,534  현재 기준=8,384  차이=850
[Unit] natural sort 통과 ✅
[Unit] 모든 단위 검증 통과 ✅


In [3]:
# ================================================================
# 셀 02 — 메인 실행 (전체 파이프라인)
# ================================================================

# ================================================================
# DeepBind M0_BRIDGE_2048_SOURCE_LOCK_v1.1
# - 현재 완전 FEMTO 데이터 43,369 CSV를 기준으로 Source Lock
# - 기존 M0 N=7,534는 legacy partial reference로만 보존
# - 원본 수정/삭제/이동 없음
# ================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import hashlib
import json
import math
import os
import re
import traceback

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd


# ================================================================
# 1. 고정 설정
# ================================================================

VERSION = "M0_BRIDGE_2048_SOURCE_LOCK_v1.1"

FEMTO_FS = 25600
SNAPSHOT_DURATION_SEC = 0.1
EXPECTED_SAMPLES = 2560
COL_H = 4

PROJECT_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb"),
    Path("/content/drive/My Drive/Colab Notebooks/field_iis3dwb"),
]

FEMTO_ROOT_CANDIDATES = [
    Path(
        "/content/drive/MyDrive/Colab Notebooks/"
        "3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing"
    ),
    Path(
        "/content/drive/My Drive/Colab Notebooks/"
        "3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing"
    ),
]

SPLIT_DIRS = {
    "LEARNING": "Training(Learning)_set",
    "TEST": "Test(Test)_set",
    "FULL_TEST": "Validation(Full_Test)_Set",
}

# 현재 완전 데이터셋 기준
CURRENT_EXPECTED = {
    "LEARNING": {
        "Bearing1_1": 3269,
        "Bearing1_2": 1015,
        "Bearing2_1": 1062,
        "Bearing2_2": 797,
        "Bearing3_1": 604,
        "Bearing3_2": 1637,
    },
    "TEST": {
        "Bearing1_3": 590,
        "Bearing1_4": 1327,
        "Bearing1_5": 2677,
        "Bearing1_6": 2232,
        "Bearing1_7": 1752,
        "Bearing2_3": 1202,
        "Bearing2_4": 713,
        "Bearing2_5": 2337,
        "Bearing2_6": 572,
        "Bearing2_7": 200,
        "Bearing3_3": 1860,
    },
    "FULL_TEST": {
        "Bearing1_3": 2375,
        "Bearing1_4": 1665,
        "Bearing1_5": 2873,
        "Bearing1_6": 2856,
        "Bearing1_7": 2635,
        "Bearing2_3": 1955,
        "Bearing2_4": 876,
        "Bearing2_5": 2697,
        "Bearing2_6": 817,
        "Bearing2_7": 268,
        "Bearing3_3": 506,
    },
}

EXPECTED_SPLIT_TOTALS = {
    split: sum(values.values())
    for split, values in CURRENT_EXPECTED.items()
}

EXPECTED_TOTAL = sum(EXPECTED_SPLIT_TOTALS.values())

assert EXPECTED_SPLIT_TOTALS == {
    "LEARNING": 8384,
    "TEST": 15462,
    "FULL_TEST": 19523,
}
assert EXPECTED_TOTAL == 43369

# 이전 M0 실행 당시의 부분 데이터 기준
LEGACY_M0_LEARNING_N = {
    "Bearing1_1": 2803,
    "Bearing1_2": 871,
    "Bearing2_1": 911,
    "Bearing2_2": 797,
    "Bearing3_1": 515,
    "Bearing3_2": 1637,
}

REFERENCE_NOTEBOOK_CANDIDATES = [
    "M0_FEMTO_Baseline_v1_baseline고정.ipynb",
    "M0_FEMTO_Baseline_v1_baseline.ipynb",
    "M0_FEMTO_Baseline_v1.ipynb",
]

REFERENCE_FILES = [
    "BASELINE_M0_frozen.json",
    "m0_baseline_result.csv",
]


# ================================================================
# 2. 공통 함수
# ================================================================

def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None


def natural_key(value):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", str(value))
    ]


def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if np.isnan(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, set):
        return sorted(value)
    return str(value)


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")

    with tmp.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
            default=json_default,
        )

    os.replace(tmp, path)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def quick_fingerprint(path, block_size=8192):
    """
    전체 43,369개 파일을 매번 full SHA로 읽지 않기 위한 빠른 지문.
    quick fingerprint가 같은 파일만 full SHA-256으로 재검증한다.
    """
    path = Path(path)
    size = path.stat().st_size

    digest = hashlib.sha256()
    digest.update(str(size).encode("ascii"))

    with path.open("rb") as file:
        head = file.read(block_size)
        digest.update(head)

        if size > block_size:
            file.seek(max(0, size - block_size))
            tail = file.read(block_size)
            digest.update(tail)

    return digest.hexdigest()


def valid_csv(path):
    path = Path(path)

    if not path.is_file():
        return False, "NOT_FILE"

    if path.suffix.lower() != ".csv":
        return False, "NOT_CSV"

    if path.name.startswith((".", "~")):
        return False, "TEMP_FILE"

    if path.stat().st_size <= 0:
        return False, "EMPTY_FILE"

    return True, "OK"


def locate_reference(project_root, names):
    for name in names:
        direct = project_root / name
        if direct.exists():
            return direct

    found = []
    for name in names:
        found.extend(project_root.rglob(name))

    found = sorted(
        set(found),
        key=lambda path: (
            len(path.parts),
            natural_key(str(path)),
        ),
    )

    return found[0] if found else None


def safe_csv_read(path):
    """
    FEMTO CSV에 header가 있거나 없는 경우 모두 처리한다.
    반환: raw_df, signal, data_row_count, header_detected
    """
    frame = pd.read_csv(
        path,
        header=None,
        low_memory=False,
        on_bad_lines="error",
    )

    if frame.shape[1] <= COL_H:
        raise ValueError(
            f"COL_H={COL_H} 없음: columns={frame.shape[1]}"
        )

    signal = pd.to_numeric(
        frame.iloc[:, COL_H],
        errors="coerce",
    )

    header_detected = False

    # 첫 행만 비수치이고 이후가 수치이면 header 1행으로 판단
    if (
        len(signal) == EXPECTED_SAMPLES + 1
        and pd.isna(signal.iloc[0])
        and signal.iloc[1:].notna().all()
    ):
        header_detected = True
        signal = signal.iloc[1:].reset_index(drop=True)

    return frame, signal, len(signal), header_detected


# ================================================================
# 3. 경로 및 출력 폴더
# ================================================================

PROJECT_ROOT = first_existing(PROJECT_ROOT_CANDIDATES)
FEMTO_ROOT = first_existing(FEMTO_ROOT_CANDIDATES)

if PROJECT_ROOT is None:
    raise FileNotFoundError("PROJECT_ROOT_NOT_FOUND")

if FEMTO_ROOT is None:
    raise FileNotFoundError("FEMTO_ROOT_NOT_FOUND")

RUN_ID = datetime.now().strftime(
    "%Y%m%d_%H%M%S_source_lock_v1_1"
)

OUTPUT_DIR = (
    PROJECT_ROOT / "bridge_outputs" / RUN_ID
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

# 실행 폴더와 분리된 영구 checkpoint
CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "bridge_outputs"
    / "_checkpoints"
    / "source_lock_v1_1"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

FINGERPRINT_CHECKPOINT = (
    CHECKPOINT_DIR / "fingerprint_checkpoint.csv"
)

print("=" * 72)
print("VERSION      :", VERSION)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("FEMTO_ROOT   :", FEMTO_ROOT)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("EXPECTED     :", EXPECTED_SPLIT_TOTALS)
print("TOTAL        :", EXPECTED_TOTAL)
print("=" * 72)


# ================================================================
# 4. 감사 Gate
# ================================================================

audit_candidates = sorted(
    PROJECT_ROOT.glob(
        "audit_outputs/*/FINAL_AUDIT_SUMMARY_v1_1.json"
    ),
    key=lambda path: path.stat().st_mtime_ns,
    reverse=True,
)

audit_gate = {
    "found": False,
    "gate_pass": False,
    "reason": "NOT_FOUND",
}

if audit_candidates:
    audit_path = audit_candidates[0]

    try:
        with audit_path.open("r", encoding="utf-8") as file:
            audit_data = json.load(file)

        final_status = audit_data.get("final_status")
        next_action_audit = audit_data.get(
            "recommended_next_action"
        )
        unchanged = audit_data.get(
            "source_files_unchanged"
        )

        audit_pass = (
            final_status in {
                "AUDIT_PASS",
                "AUDIT_PASS_WITH_CRITERIA_WARNING",
            }
            and next_action_audit == "READY_FOR_M0_BRIDGE_2048"
            and unchanged is True
        )

        audit_gate = {
            "found": True,
            "path": str(audit_path),
            "sha256": sha256_file(audit_path),
            "final_status": final_status,
            "recommended_next_action": next_action_audit,
            "source_files_unchanged": unchanged,
            "gate_pass": audit_pass,
            "reason": "OK" if audit_pass
            else "AUDIT_CONDITION_NOT_MET",
        }

    except Exception as error:
        audit_gate = {
            "found": True,
            "path": str(audit_path),
            "gate_pass": False,
            "reason": f"PARSE_ERROR:{error}",
        }

write_json(
    OUTPUT_DIR / "audit_gate_check.json",
    audit_gate,
)

print(f"[GATE] audit_gate_pass={audit_gate['gate_pass']}  "
      f"final_status={audit_gate.get('final_status')}  "
      f"source_files_unchanged={audit_gate.get('source_files_unchanged')}")


# ================================================================
# 5. 기준 파일 사전 해시
# ================================================================

reference_paths = []

for name in REFERENCE_FILES:
    selected = locate_reference(PROJECT_ROOT, [name])
    if selected is not None:
        reference_paths.append(selected)

selected_notebook = locate_reference(
    PROJECT_ROOT,
    REFERENCE_NOTEBOOK_CANDIDATES,
)

if selected_notebook is not None:
    reference_paths.append(selected_notebook)

if audit_gate.get("path"):
    reference_paths.append(Path(audit_gate["path"]))

reference_before = {}

for path in reference_paths:
    stat = path.stat()

    reference_before[str(path.resolve())] = {
        "file_name": path.name,
        "size": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
        "sha256": sha256_file(path),
    }

print("Selected reference notebook:", selected_notebook)
print(f"기준 파일 {len(reference_before)}개 사전 해시 완료")


# ================================================================
# 6. CSV Manifest 생성
# ================================================================

manifest_rows = []
count_rows = []
scan_errors = []

for split, directory_name in SPLIT_DIRS.items():
    split_path = FEMTO_ROOT / directory_name

    if not split_path.exists():
        scan_errors.append({
            "scope": split,
            "path": str(split_path),
            "error": "SPLIT_DIRECTORY_NOT_FOUND",
        })
        continue

    bearing_dirs = sorted(
        [
            path for path in split_path.iterdir()
            if path.is_dir()
            and re.fullmatch(r"Bearing\d+_\d+", path.name)
        ],
        key=lambda path: natural_key(path.name),
    )

    for bearing_dir in bearing_dirs:
        record_uid = f"{split}/{bearing_dir.name}"

        csv_files = []

        try:
            for path in bearing_dir.iterdir():
                ok, reason = valid_csv(path)

                if ok:
                    csv_files.append(path)

            csv_files.sort(
                key=lambda path: natural_key(path.name)
            )

        except Exception as error:
            scan_errors.append({
                "scope": record_uid,
                "path": str(bearing_dir),
                "error": repr(error),
            })
            continue

        expected = CURRENT_EXPECTED.get(
            split, {}
        ).get(bearing_dir.name)

        actual = len(csv_files)

        count_rows.append({
            "split": split,
            "record_uid": record_uid,
            "logical_bearing_id": bearing_dir.name,
            "expected_current_complete_count": expected,
            "actual_csv_count": actual,
            "count_difference":
                None if expected is None else actual - expected,
            "count_match":
                expected is not None and actual == expected,
            "legacy_m0_partial_N":
                LEGACY_M0_LEARNING_N.get(bearing_dir.name)
                if split == "LEARNING" else None,
            "legacy_difference":
                (
                    actual
                    - LEGACY_M0_LEARNING_N[bearing_dir.name]
                    if split == "LEARNING"
                    and bearing_dir.name in LEGACY_M0_LEARNING_N
                    else None
                ),
            "status":
                "PASS_CURRENT_COMPLETE"
                if expected == actual
                else "CURRENT_COUNT_MISMATCH",
        })

        for index, path in enumerate(csv_files):
            try:
                stat = path.stat()

                manifest_rows.append({
                    "version": VERSION,
                    "split": split,
                    "record_uid": record_uid,
                    "logical_bearing_id": bearing_dir.name,
                    "sequence_index": index,
                    "csv_file_name": path.name,
                    "absolute_path": str(path.resolve()),
                    "relative_path": str(
                        path.relative_to(FEMTO_ROOT)
                    ),
                    "file_size_bytes": int(stat.st_size),
                    "mtime_ns": int(stat.st_mtime_ns),
                    "scan_status": "OK",
                })

            except Exception as error:
                scan_errors.append({
                    "scope": record_uid,
                    "path": str(path),
                    "error": repr(error),
                })

MANIFEST_DF = pd.DataFrame(manifest_rows)
COUNT_DF = pd.DataFrame(count_rows)
ERROR_DF = pd.DataFrame(scan_errors)

MANIFEST_DF.to_csv(
    OUTPUT_DIR / "femto_csv_source_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

COUNT_DF.to_csv(
    OUTPUT_DIR / "split_csv_count_check.csv",
    index=False,
    encoding="utf-8-sig",
)

ERROR_DF.to_csv(
    OUTPUT_DIR / "source_scan_errors.csv",
    index=False,
    encoding="utf-8-sig",
)

write_json(
    OUTPUT_DIR / "femto_csv_source_manifest.json",
    MANIFEST_DF.to_dict(orient="records"),
)

print(f"[MANIFEST] 총 {len(MANIFEST_DF):,}개 CSV 스캔 완료, scan_errors={len(scan_errors)}")


# ================================================================
# 7. Quick fingerprint + checkpoint
# ================================================================

cache = {}

if FINGERPRINT_CHECKPOINT.exists():
    try:
        old = pd.read_csv(FINGERPRINT_CHECKPOINT)

        for row in old.to_dict(orient="records"):
            key = (
                str(row["absolute_path"]),
                int(float(row["file_size_bytes"])),
                int(float(row["mtime_ns"])),
            )
            cache[key] = row.get("quick_fingerprint")

        print(f"[FINGERPRINT] checkpoint 로드: {len(cache):,}개 캐시")

    except Exception as err:
        print(f"[FINGERPRINT] checkpoint 로드 실패 (무시): {err}")
        cache = {}

fingerprint_rows = []
fingerprint_errors = []
total_files = len(MANIFEST_DF)

for number, row in enumerate(
    MANIFEST_DF.to_dict(orient="records"),
    start=1,
):
    key = (
        row["absolute_path"],
        int(row["file_size_bytes"]),
        int(row["mtime_ns"]),
    )

    fingerprint = cache.get(key)

    if not isinstance(fingerprint, str) or not fingerprint:
        try:
            fingerprint = quick_fingerprint(
                row["absolute_path"]
            )
        except Exception as error:
            fingerprint = None
            fingerprint_errors.append({
                "absolute_path": row["absolute_path"],
                "error": repr(error),
            })

    fingerprint_rows.append({
        **row,
        "quick_fingerprint": fingerprint,
    })

    cache[key] = fingerprint

    if number % 500 == 0:
        checkpoint_df = pd.DataFrame([
            {
                "absolute_path": k[0],
                "file_size_bytes": k[1],
                "mtime_ns": k[2],
                "quick_fingerprint": v,
            }
            for k, v in cache.items()
        ])

        tmp_path = FINGERPRINT_CHECKPOINT.with_suffix(
            ".csv.tmp"
        )

        checkpoint_df.to_csv(
            tmp_path,
            index=False,
            encoding="utf-8-sig",
        )

        os.replace(tmp_path, FINGERPRINT_CHECKPOINT)

        print(
            f"[FINGERPRINT] {number:,}/{total_files:,}"
        )

# 최종 checkpoint 저장
final_chk_df = pd.DataFrame([
    {"absolute_path": k[0], "file_size_bytes": k[1],
     "mtime_ns": k[2], "quick_fingerprint": v}
    for k, v in cache.items()
])
final_chk_df.to_csv(FINGERPRINT_CHECKPOINT, index=False, encoding="utf-8-sig")

FINGERPRINT_DF = pd.DataFrame(fingerprint_rows)

FINGERPRINT_DF.to_csv(
    OUTPUT_DIR / "source_fingerprint_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame(fingerprint_errors).to_csv(
    OUTPUT_DIR / "fingerprint_errors.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"[FINGERPRINT] 완료: {len(FINGERPRINT_DF):,}개, errors={len(fingerprint_errors)}")


# ================================================================
# 8. 충돌 후보만 Full SHA-256
# ================================================================

valid_fp = FINGERPRINT_DF[
    FINGERPRINT_DF["quick_fingerprint"].notna()
].copy()

collision_counts = (
    valid_fp["quick_fingerprint"].value_counts()
)

collision_fingerprints = set(
    collision_counts[collision_counts > 1].index
)

collision_df = valid_fp[
    valid_fp["quick_fingerprint"].isin(
        collision_fingerprints
    )
].copy()

print(f"[FULL SHA] fingerprint 충돌 후보: {len(collision_df):,}개")

full_sha_rows = []

for row in collision_df.to_dict(orient="records"):
    try:
        full_sha = sha256_file(row["absolute_path"])
        status = "OK"
        error_message = None
    except Exception as error:
        full_sha = None
        status = "HASH_ERROR"
        error_message = repr(error)

    full_sha_rows.append({
        **row,
        "sha256": full_sha,
        "hash_status": status,
        "error_message": error_message,
    })

FULL_SHA_DF = pd.DataFrame(full_sha_rows)

FULL_SHA_DF.to_csv(
    OUTPUT_DIR / "collision_full_sha256.csv",
    index=False,
    encoding="utf-8-sig",
)

within_duplicate_rows = []
cross_split_rows = []

if not FULL_SHA_DF.empty:
    for sha256, group in FULL_SHA_DF.dropna(
        subset=["sha256"]
    ).groupby("sha256"):

        if len(group) <= 1:
            continue

        record_uids = sorted(
            group["record_uid"].unique()
        )

        for record_uid, record_group in group.groupby(
            "record_uid"
        ):
            if len(record_group) > 1:
                within_duplicate_rows.append({
                    "record_uid": record_uid,
                    "sha256": sha256,
                    "duplicate_count": len(record_group),
                    "file_names": "|".join(
                        record_group["csv_file_name"]
                    ),
                    "status":
                        "DUPLICATE_CONTENT_WITHIN_RECORD",
                })

        if len(record_uids) > 1:
            learning_involved = any(
                uid.startswith("LEARNING/")
                for uid in record_uids
            )

            cross_split_rows.append({
                "sha256": sha256,
                "occurrence_count": len(group),
                "record_uids": "|".join(record_uids),
                "learning_involved": learning_involved,
                "status":
                    "UNEXPECTED_LEARNING_CROSS_SPLIT_OVERLAP"
                    if learning_involved
                    else "KNOWN_OR_EXPECTED_CROSS_SPLIT_OVERLAP",
                "blocking": learning_involved,
            })

WITHIN_DUP_DF = pd.DataFrame(within_duplicate_rows)
CROSS_DUP_DF = pd.DataFrame(cross_split_rows)

WITHIN_DUP_DF.to_csv(
    OUTPUT_DIR / "csv_duplicate_hash_within_record.csv",
    index=False,
    encoding="utf-8-sig",
)

CROSS_DUP_DF.to_csv(
    OUTPUT_DIR / "csv_duplicate_hash_cross_split.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"[DUP] within-record={len(WITHIN_DUP_DF)}, cross-split={len(CROSS_DUP_DF)}")


# ================================================================
# 9. Learning 30개 구조 Probe
# ================================================================

probe_rows = []

learning_manifest = MANIFEST_DF[
    MANIFEST_DF["split"] == "LEARNING"
].copy()

for record_uid, group in learning_manifest.groupby(
    "record_uid"
):
    group = group.sort_values("sequence_index")
    file_count = len(group)

    if file_count == 0:
        continue

    positions = sorted(set([
        0,
        int(round((file_count - 1) * 0.25)),
        int(round((file_count - 1) * 0.50)),
        int(round((file_count - 1) * 0.75)),
        file_count - 1,
    ]))

    labels = {
        positions[0]: "FIRST",
        positions[-1]: "LAST",
    }

    for position in positions:
        row = group.iloc[position]
        path = Path(row["absolute_path"])

        result = {
            "record_uid": record_uid,
            "probe_position": labels.get(
                position, f"INDEX_{position}"
            ),
            "sequence_index": int(
                row["sequence_index"]
            ),
            "csv_file_name": path.name,
            "absolute_path": str(path),
            "probe_status": "ERROR",
            "error_message": None,
        }

        try:
            frame, signal, row_count, header_detected = (
                safe_csv_read(path)
            )

            finite = signal.replace(
                [np.inf, -np.inf],
                np.nan,
            )

            nan_count = int(finite.isna().sum())
            inf_count = int(
                np.isinf(
                    pd.to_numeric(
                        signal,
                        errors="coerce",
                    ).to_numpy(dtype=float)
                ).sum()
            )

            numeric = finite.dropna().to_numpy(
                dtype=float
            )

            constant_signal = bool(
                len(numeric) > 0
                and np.std(numeric) == 0
            )

            passed = (
                row_count == EXPECTED_SAMPLES
                and frame.shape[1] > COL_H
                and nan_count == 0
                and inf_count == 0
                and len(numeric) == EXPECTED_SAMPLES
                and not constant_signal
            )

            result.update({
                "raw_row_count": int(frame.shape[0]),
                "data_row_count": int(row_count),
                "column_count": int(frame.shape[1]),
                "header_detected": header_detected,
                "selected_column_index": COL_H,
                "nan_count": nan_count,
                "inf_count": inf_count,
                "constant_signal": constant_signal,
                "rms": float(
                    np.sqrt(np.mean(numeric ** 2))
                ) if len(numeric) else None,
                "minimum": float(np.min(numeric))
                if len(numeric) else None,
                "maximum": float(np.max(numeric))
                if len(numeric) else None,
                "standard_deviation": float(
                    np.std(numeric)
                ) if len(numeric) else None,
                "probe_status": "PASS"
                if passed else "FAIL",
            })

        except Exception as error:
            result["error_message"] = repr(error)

        probe_rows.append(result)

PROBE_DF = pd.DataFrame(probe_rows)

PROBE_DF.to_csv(
    OUTPUT_DIR / "csv_structure_probe.csv",
    index=False,
    encoding="utf-8-sig",
)

probe_pass_count = int(
    (PROBE_DF["probe_status"] == "PASS").sum()
) if not PROBE_DF.empty else 0

print(f"[PROBE] {probe_pass_count}/{len(PROBE_DF)} PASS")


# ================================================================
# 10. 기준 파일 불변성
# ================================================================

immutability_rows = []

for path_text, before in reference_before.items():
    path = Path(path_text)

    if not path.exists():
        immutability_rows.append({
            "absolute_path": path_text,
            "file_name": before["file_name"],
            "unchanged": False,
            "status": "MISSING_AFTER",
        })
        continue

    stat = path.stat()
    after_sha = sha256_file(path)

    unchanged = (
        int(stat.st_size) == before["size"]
        and after_sha == before["sha256"]
    )

    immutability_rows.append({
        "absolute_path": path_text,
        "file_name": before["file_name"],
        "size_before": before["size"],
        "size_after": int(stat.st_size),
        "sha256_before": before["sha256"],
        "sha256_after": after_sha,
        "unchanged": unchanged,
        "status": "UNCHANGED"
        if unchanged else "MODIFIED",
    })

IMMUTABILITY_DF = pd.DataFrame(immutability_rows)

IMMUTABILITY_DF.to_csv(
    OUTPUT_DIR / "source_file_immutability_check.csv",
    index=False,
    encoding="utf-8-sig",
)


# ================================================================
# 11. 최종 판정
# ================================================================

actual_split_totals = (
    COUNT_DF.groupby("split")["actual_csv_count"]
    .sum()
    .to_dict()
)

actual_total = int(
    COUNT_DF["actual_csv_count"].sum()
)

count_pass = bool(
    len(COUNT_DF) == 28
    and COUNT_DF["count_match"].all()
    and actual_total == EXPECTED_TOTAL
)

probe_pass = bool(
    len(PROBE_DF) == 30
    and PROBE_DF["probe_status"].eq("PASS").all()
)

within_duplicate_count = len(WITHIN_DUP_DF)

unexpected_learning_overlap_count = (
    int(CROSS_DUP_DF["blocking"].sum())
    if not CROSS_DUP_DF.empty
    and "blocking" in CROSS_DUP_DF.columns
    else 0
)

immutability_pass = bool(
    not IMMUTABILITY_DF.empty
    and IMMUTABILITY_DF["unchanged"].all()
)

fingerprint_error_count = len(fingerprint_errors)
scan_error_count = len(scan_errors)

if not audit_gate.get("gate_pass", False):
    source_lock_status = (
        "SOURCE_LOCK_BLOCKED_BY_AUDIT_GATE"
    )
    next_action = "FIX_AUDIT_GATE_FIRST"

elif not count_pass:
    source_lock_status = (
        "SOURCE_LOCK_HOLD_CURRENT_COUNT_MISMATCH"
    )
    next_action = "REVIEW_SOURCE_INVENTORY"

elif scan_error_count > 0 or fingerprint_error_count > 0:
    source_lock_status = (
        "SOURCE_LOCK_HOLD_IO_ERRORS"
    )
    next_action = "RERUN_WITH_CHECKPOINT"

elif within_duplicate_count > 0:
    source_lock_status = (
        "SOURCE_LOCK_HOLD_DUPLICATE_CONTENT"
    )
    next_action = "MANUAL_DUPLICATE_REVIEW"

elif unexpected_learning_overlap_count > 0:
    source_lock_status = (
        "SOURCE_LOCK_MANUAL_REVIEW_CROSS_SPLIT_OVERLAP"
    )
    next_action = "MANUAL_CROSS_SPLIT_REVIEW"

elif not probe_pass:
    source_lock_status = (
        "SOURCE_LOCK_HOLD_SCHEMA_MISMATCH"
    )
    next_action = "REVIEW_CSV_SCHEMA"

elif not immutability_pass:
    source_lock_status = (
        "SOURCE_LOCK_INVALID_SOURCE_MODIFIED"
    )
    next_action = "MANUAL_REVIEW_REQUIRED"

else:
    source_lock_status = "SOURCE_LOCK_PASS"
    next_action = "READY_FOR_M0_BRIDGE_2048_PREP"


summary = {
    "source_lock_version": VERSION,
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "source_lock_status": source_lock_status,
    "recommended_next_action": next_action,
    "audit_gate_pass": audit_gate.get(
        "gate_pass", False
    ),
    "project_root": str(PROJECT_ROOT),
    "femto_root": str(FEMTO_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "current_complete_dataset": {
        "expected_split_totals":
            EXPECTED_SPLIT_TOTALS,
        "actual_split_totals":
            actual_split_totals,
        "expected_total": EXPECTED_TOTAL,
        "actual_total": actual_total,
        "count_pass": count_pass,
    },
    "legacy_m0_partial_reference": {
        "learning_expected_total": sum(
            LEGACY_M0_LEARNING_N.values()
        ),
        "current_learning_total":
            actual_split_totals.get("LEARNING"),
        "difference":
            actual_split_totals.get("LEARNING", 0)
            - sum(LEGACY_M0_LEARNING_N.values()),
        "classification":
            "LEGACY_M0_PARTIAL_REFERENCE",
        "blocking": False,
    },
    "probe": {
        "expected_count": 30,
        "actual_count": len(PROBE_DF),
        "pass_count": probe_pass_count,
        "pass": probe_pass,
    },
    "duplicate_checks": {
        "within_record_duplicate_groups":
            within_duplicate_count,
        "unexpected_learning_cross_split_groups":
            unexpected_learning_overlap_count,
    },
    "io_errors": {
        "scan_errors": scan_error_count,
        "fingerprint_errors":
            fingerprint_error_count,
    },
    "source_files_unchanged":
        immutability_pass,
    "selected_reference_notebook":
        str(selected_notebook)
        if selected_notebook else None,
    "bridge_conversion_performed": False,
    "original_files_modified": False,
}

write_json(
    OUTPUT_DIR / "SOURCE_LOCK_SUMMARY.json",
    summary,
)

summary_md = f"""# M0 Bridge Source Lock v1.1

- **Status:** `{source_lock_status}`
- **Next action:** `{next_action}`
- **Current complete CSV:** `{actual_total:,} / {EXPECTED_TOTAL:,}`
- **Learning:** `{actual_split_totals.get("LEARNING", 0):,}`
- **Test:** `{actual_split_totals.get("TEST", 0):,}`
- **Full Test:** `{actual_split_totals.get("FULL_TEST", 0):,}`
- **Legacy M0 Learning N:** `{sum(LEGACY_M0_LEARNING_N.values()):,}`
- **Legacy/current difference:** `{actual_split_totals.get("LEARNING", 0) - sum(LEGACY_M0_LEARNING_N.values()):,}`
- **Probe:** `{probe_pass_count}/30`
- **Within-record duplicate groups:** `{within_duplicate_count}`
- **Unexpected Learning cross-split groups:** `{unexpected_learning_overlap_count}`
- **Source immutability:** `{immutability_pass}`
- **Output:** `{OUTPUT_DIR}`

## Interpretation

기존 M0의 Learning N=7,534는 이전 부분 데이터 기준으로 보존한다.
현재 완전 FEMTO 데이터셋은 43,369개 CSV이며 Source Lock은 현재
완전 데이터셋을 대상으로 한다. 기존 M0 frozen 결과는 수정하지 않는다.
"""

with (
    OUTPUT_DIR / "SOURCE_LOCK_SUMMARY.md"
).open("w", encoding="utf-8") as file:
    file.write(summary_md)

print("\n" + "=" * 72)
print("SOURCE LOCK STATUS  :", source_lock_status)
print("NEXT ACTION         :", next_action)
print(
    "CURRENT CSV TOTAL   :",
    f"{actual_total:,}/{EXPECTED_TOTAL:,}",
)
print(
    "SPLIT TOTALS        :",
    actual_split_totals,
)
print(
    "LEGACY M0 LEARNING  :",
    f"{sum(LEGACY_M0_LEARNING_N.values()):,}",
)
print(
    "PROBE PASS          :",
    f"{probe_pass_count}/30",
)
print(
    "DUPLICATE GROUPS    :",
    within_duplicate_count,
)
print(
    "SOURCE UNCHANGED    :",
    immutability_pass,
)
print("OUTPUT DIR          :", OUTPUT_DIR)
print("=" * 72)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
VERSION      : M0_BRIDGE_2048_SOURCE_LOCK_v1.1
PROJECT_ROOT : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb
FEMTO_ROOT   : /content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing
OUTPUT_DIR   : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_150815_source_lock_v1_1
EXPECTED     : {'LEARNING': 8384, 'TEST': 15462, 'FULL_TEST': 19523}
TOTAL        : 43369
[GATE] audit_gate_pass=True  final_status=AUDIT_PASS_WITH_CRITERIA_WARNING  source_files_unchanged=True
Selected reference notebook: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/M0_FEMTO_Baseline_v1_baseline고정.ipynb
기준 파일 4개 사전 해시 완료
[MANIFEST] 총 43,369개 CSV 스캔 완료, scan_errors=0
[FINGERPRINT] checkpoint 로드: 43,369개 캐시
[FINGERPRINT] 500/43,369
[FINGERPRINT] 1,000/43,369
[FINGERPRINT] 1,500/43,369
[FINGERPRINT] 2,000/43,369
[FINGERPRINT] 

In [4]:
# ================================================================
# 셀 03 — fatal 오류 확인 (셀 02 실패 시 여기서 확인)
# ================================================================

# 셀 02가 예외로 중단됐을 때 OUTPUT_DIR에 fatal_error.json을 생성했는지 확인
# 이 셀은 셀 02와 독립적으로 실행 가능

import json
import traceback
from pathlib import Path
from datetime import datetime

# OUTPUT_DIR이 정의돼 있는 경우만 확인
try:
    _od = OUTPUT_DIR
except NameError:
    print("OUTPUT_DIR 미정의 — 셀 02를 먼저 실행하세요.")
    _od = None

if _od is not None:
    fatal_path = Path(_od) / "fatal_error.json"
    if fatal_path.exists():
        with fatal_path.open("r", encoding="utf-8") as f:
            err_data = json.load(f)
        print("🚨 fatal_error.json 발견:")
        print(json.dumps(err_data, ensure_ascii=False, indent=2))
    else:
        # SOURCE_LOCK_SUMMARY.json이 있으면 최종 상태 출력
        summary_path = Path(_od) / "SOURCE_LOCK_SUMMARY.json"
        if summary_path.exists():
            with summary_path.open("r", encoding="utf-8") as f:
                s = json.load(f)
            print("=" * 60)
            print(f"SOURCE_LOCK_STATUS  : {s.get('source_lock_status')}")
            print(f"NEXT ACTION         : {s.get('recommended_next_action')}")
            print(f"CURRENT TOTAL       : {s.get('current_complete_dataset', {}).get('actual_total')}")
            print(f"SOURCE UNCHANGED    : {s.get('source_files_unchanged')}")
            print("=" * 60)
        else:
            print("SOURCE_LOCK_SUMMARY.json 없음 — 셀 02 실행 중 또는 미완료 상태")

SOURCE_LOCK_STATUS  : SOURCE_LOCK_HOLD_SCHEMA_MISMATCH
NEXT ACTION         : REVIEW_CSV_SCHEMA
CURRENT TOTAL       : 43369
SOURCE UNCHANGED    : True
